In [1]:
import json
from ibm_watson import NaturalLanguageUnderstandingV1
from ibm_watson.natural_language_understanding_v1 import Features, SentimentOptions, EntitiesOptions, CategoriesOptions
from ibm_cloud_sdk_core.authenticators import IAMAuthenticator

In [3]:
from api_key import IBM_API_KEY, IBM_URL

In [4]:
TEXT_INPUT = "The new iPhone has great features but is too expensive for most people."


In [5]:
def init_ibm_nlu():
    authenticator = IAMAuthenticator(IBM_API_KEY)
    nlu = NaturalLanguageUnderstandingV1(
        version='2021-08-01',
        authenticator=authenticator
    )
    nlu.set_service_url(IBM_URL)
    return nlu

In [6]:
nlu_client = init_ibm_nlu()

In [17]:
def analyze_ibm_sentiment(text):
    response = nlu_client.analyze(
        text=text,
        features=Features(sentiment=SentimentOptions())
    ).get_result()
    return response

In [18]:
analyze_ibm_sentiment(TEXT_INPUT)

{'usage': {'text_units': 1, 'text_characters': 71, 'features': 1},
 'sentiment': {'document': {'score': -0.637434, 'label': 'negative'}},
 'language': 'en'}

In [9]:
def analyze_ibm_entities(text):
    response = nlu_client.analyze(
        text=text,
        features=Features(entities=EntitiesOptions(emotion=False, sentiment=True, limit=10))
    ).get_result()
    return response

In [19]:
analyze_ibm_entities(TEXT_INPUT)

{'usage': {'text_units': 1, 'text_characters': 71, 'features': 1},
 'language': 'en',
 'entities': [{'type': 'Organization',
   'text': 'iPhone',
   'sentiment': {'score': -0.637434, 'label': 'negative'},
   'relevance': 0.978348,
   'count': 1,
   'confidence': 0.660147}]}

In [11]:
def classify_ibm_text(text):
    response = nlu_client.analyze(
        text=text,
        features=Features(categories=CategoriesOptions())
    ).get_result()
    return response

In [12]:
classify_ibm_text(TEXT_INPUT)

{'usage': {'text_units': 1, 'text_characters': 71, 'features': 1},
 'language': 'en',
 'categories': [{'score': 0.873581,
   'label': '/technology and computing/tech news'},
  {'score': 0.750581, 'label': '/technology and computing/operating systems'},
  {'score': 0.736367, 'label': '/technology and computing/hardware'}]}

In [ ]:
import json

def analyze_and_store(data, content_type, results):
    for entry in data.get(content_type, []):
        try:
            sentiment_response = analyze_ibm_sentiment(entry)
        except Exception as e:
            print(f"Error analyzing sentiment for {content_type}: {e}")
            sentiment_response = None
        try:
            entities_response = analyze_ibm_entities(entry)
        except Exception as e:
            print(f"Error analyzing entities for {content_type}: {e}")
            entities_response = None
        try:
            categories_response = classify_ibm_text(entry)
        except Exception as e:
            print(f"Error classifying categories for {content_type}: {e}")
            categories_response = None

        print(f"Sentiment ({content_type}):", sentiment_response)
        print("Entities:", entities_response)
        print("Categories:", categories_response)
        print("-----------------------------------------------------")

        results.append({
            'type': content_type,
            content_type: entry,
            'sentiment': sentiment_response,
            'entities': entities_response,
            'categories': categories_response
        })

# Main
with open('./data/input.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

results = []
for section in ['articles', 'comments', 'reviews']:
    print(f"Analyzing {section}...")
    analyze_and_store(data, section, results)

with open('./data/ibm_output.json', 'w', encoding='utf-8') as output_file:
    json.dump(results, output_file, ensure_ascii=False, indent=4)


Analyzing articles...
Sentiment (articles): {'usage': {'text_units': 1, 'text_characters': 2337, 'features': 1}, 'sentiment': {'document': {'score': -0.588211, 'mixed': '1', 'label': 'negative'}}, 'language': 'en'}
Entities: {'usage': {'text_units': 1, 'text_characters': 2337, 'features': 1}, 'language': 'en', 'entities': [{'type': 'Money', 'text': '$800', 'sentiment': {'score': -0.312825, 'mixed': '1', 'label': 'negative'}, 'relevance': 0.95348, 'count': 4, 'confidence': 1.0}, {'type': 'Organization', 'text': 'DHL', 'sentiment': {'score': -0.548342, 'label': 'negative'}, 'relevance': 0.58845, 'disambiguation': {'name': 'DHL_Express', 'dbpedia_resource': 'http://dbpedia.org/resource/DHL_Express'}, 'count': 3, 'confidence': 0.999723}, {'type': 'Money', 'text': '$2,500', 'sentiment': {'score': -0.624395, 'label': 'negative'}, 'relevance': 0.453568, 'count': 1, 'confidence': 0.997781}, {'type': 'Date', 'text': 'earlier this month', 'sentiment': {'score': -0.624395, 'label': 'negative'}, '